In [ ]:
import sys
if '/home/ec2-user/sagemaker-pipe-template' not in sys.path:
    sys.path.insert(0, '/home/ec2-user/sagemaker-pipe-template')
import utils, boto3, paths, time
import pipeline.baseline as baseline
from sagemaker.core.helper.session_helper import Session
import pandas as pd
pd.set_option('display.max_colwidth', None)  # show full text in each cell
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.width', None)         # don't wrap

In [6]:
region = 'us-east-1'
model_package_group_name='abalone'
model_version=1
data_bucket='omm-test-bucket'
project_path = 'models/abalone'
account='088461143167'

role = utils.get_sm_service_role_arn()
boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)
sagemaker_session=Session(boto_session=boto_session)
p=paths.Paths(bucket_name='omm-test-bucket', project_name='abalone', model_prefix='abalone')

[06/15/26 20:26:41] INFO     Found credentials from IAM      ]8;id=9493386;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=9493387;file:///home/ec2-user/.local/lib/python3.9/site-packages/botocore/credentials.py#1171\1171]8;;\
                             Role: ec2-1                                        


In [ ]:
baseliner = baseline.Baseliner('abalone-1', p.data_dir, p.baseline_file, p.train_file, 'ml.m5.large', sagemaker_session)
baseliner.make_baseline_sets('rings', 'rings_prediction', target_type=float)

In [ ]:
def endpoint_exists(sm_client, endpoint_name):
    try:
        sm_client.describe_endpoint(EndpointName=endpoint_name)
        return True
    except sm_client.exceptions.ClientError:
        return False


def endpoint_config_exists(sm_client, config_name):
    try:
        sm_client.describe_endpoint_config(EndpointConfigName=config_name)
        return True
    except sm_client.exceptions.ClientError:
        return False


def handler(event, context):
    sm_client = boto3.client('sagemaker')
    
    model_name = event['model_name']
    endpoint_name = event['endpoint_name']
    instance_type = event['instance_type']
    data_capture_path = event['data_capture_path'] # 's3://omm-test-bucket/data-capture/abalone'
    endpoint_config_name = endpoint_name + "-config"

    # Create or select endpoint
    if not endpoint_config_exists(sm_client, endpoint_config_name):
        print("creating endpoint config")
        response = sm_client.create_endpoint_config(
            EndpointConfigName=endpoint_config_name,
            ProductionVariants=[
                {
                    'VariantName': 'AllTraffic',
                    'ModelName': model_name,
                    'InstanceType': instance_type,
                    'InitialInstanceCount': 1,
                    'InitialVariantWeight': 1.0
                }
            ],
            DataCaptureConfig={
                'EnableCapture': True,
                'InitialSamplingPercentage': 100,
                'DestinationS3Uri': data_capture_path,
                'CaptureOptions': [
                    {'CaptureMode': 'Input'},
                    {'CaptureMode': 'Output'}
                ]
            }
        )
        while not endpoint_config_exists(sm_client, endpoint_config_name):
            time.sleep(5)
        print(response['EndpointConfigArn'])
    else:
        print(f"using existing endpoint config {endpoint_config_name}")

    # Create or update endpoint
    if endpoint_exists(sm_client, endpoint_name):
        print("updating endpoint")
        response = sm_client.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
        print(response)
    else:
        print("creating endpoint")
        response = sm_client.create_endpoint(EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name)
        print(response)

    # Wait for endpoint to be InService
    waiter = sm_client.get_waiter('endpoint_in_service')
    waiter.wait(EndpointName=endpoint_name)
    
    return {'endpoint_name': endpoint_name}

In [ ]:
event={
    'model_name':'abalone-1',
    'endpoint_name':'abalone-endpoint',
    'instance_type':'ml.m5.large',
    'data_capture_path':p.data_capture_dir
}
context={}
handler(event, context)